# Setup and Load Data

In [3]:
"""
Notebook 03: Feature Engineering
================================
Transforms 5 raw tables into ONE modeling dataset.
Each row = one invoice at one observation date, with ~30 features + target.

CRITICAL RULE: Every feature must use ONLY information
available on or before the observation date. No future data.
"""

import numpy as np
import pandas as pd
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Load Data ──
# Change path for your environment
DATA_DIR = 'C:/Users/Manu/Desktop/Project 2 AR Collections Intelligence Engine/Raw Data/'
customers = pd.read_csv(f'{DATA_DIR}customers.csv', parse_dates=['customer_since'])
invoices = pd.read_csv(f'{DATA_DIR}invoices.csv', parse_dates=['invoice_date', 'due_date'])
payments = pd.read_csv(f'{DATA_DIR}payments.csv', parse_dates=['payment_date'])
disputes = pd.read_csv(f'{DATA_DIR}disputes.csv', parse_dates=['dispute_date', 'resolution_date'])
dunning = pd.read_csv(f'{DATA_DIR}dunning_contacts.csv', parse_dates=['contact_date', 'promised_pay_date'])

OBSERVATION_END = pd.Timestamp('2023-12-31')

print("Data loaded:")
for name, df in [('customers', customers), ('invoices', invoices), ('payments', payments),
                  ('disputes', disputes), ('dunning', dunning)]:
    print(f"  {name}: {len(df):,} rows")

Data loaded:
  customers: 5,000 rows
  invoices: 198,622 rows
  payments: 148,869 rows
  disputes: 15,719 rows
  dunning: 592,356 rows


# Data Cleaning

In [4]:
"""
STEP 1: DATA CLEANING
- Write off stale Open invoices (180+ DPD) — realistic business practice
- This fixes the inflated DSO issue found in EDA
"""

# Write off invoices past 180 DPD still marked as Open
days_open = (OBSERVATION_END - invoices['due_date']).dt.days
stale_mask = (invoices['invoice_status'] == 'Open') & (days_open > 180)
stale_count = stale_mask.sum()
invoices.loc[stale_mask, 'invoice_status'] = 'Written Off'

print(f"Written off {stale_count:,} stale invoices (180+ DPD)")
print(f"\nUpdated status distribution:")
print(invoices['invoice_status'].value_counts())
print(f"\nUpdated status (%):")
print((invoices['invoice_status'].value_counts(normalize=True) * 100).round(1))

Written off 33,710 stale invoices (180+ DPD)

Updated status distribution:
invoice_status
Paid              142068
Written Off        33736
Open               19098
Partially Paid      2972
Disputed             748
Name: count, dtype: int64

Updated status (%):
invoice_status
Paid              71.5
Written Off       17.0
Open               9.6
Partially Paid     1.5
Disputed           0.4
Name: proportion, dtype: float64


# Build the Observation Date Framework

In [5]:
"""
STEP 2: OBSERVATION DATE FRAMEWORK

This is the most important concept in the whole feature engineering step.

Instead of predicting once per invoice, we create MULTIPLE observation
points per invoice. At each point, we ask:
  "Given everything known AS OF THIS DATE, will this invoice be paid within 90 days?"

This prevents leakage because features are always computed
relative to the observation date, not the invoice outcome.

Observation dates:
  - due_date (Day 0: invoice just became due)
  - due_date + 15 days
  - due_date + 30 days
  - due_date + 60 days

We ONLY create observations where:
  1. The observation date is before the end of our data window
  2. The invoice was still "active" (not already paid/written off before that date)
"""

# Pre-compute: first full payment date per invoice
# (sum payments per invoice — if total >= invoice amount, it's fully paid)
payment_agg = payments.groupby('invoice_id').agg(
    first_payment_date=('payment_date', 'min'),
    last_payment_date=('payment_date', 'max'),
    total_paid=('payment_amount', 'sum'),
    payment_count=('payment_id', 'count'),
).reset_index()

inv_base = invoices.merge(payment_agg, on='invoice_id', how='left')
inv_base['total_paid'] = inv_base['total_paid'].fillna(0)
inv_base['is_fully_paid'] = inv_base['total_paid'] >= (inv_base['invoice_amount'] * 0.99)
inv_base['full_pay_date'] = inv_base.apply(
    lambda r: r['last_payment_date'] if r['is_fully_paid'] else pd.NaT, axis=1
)

# Create observation points
observation_offsets = [0, 15, 30, 60]  # days after due_date
observations = []

for offset in observation_offsets:
    temp = inv_base.copy()
    temp['observation_date'] = temp['due_date'] + timedelta(days=offset)

    # Filter 1: observation date must be within data window
    temp = temp[temp['observation_date'] <= OBSERVATION_END]

    # Filter 2: invoice must not be fully paid BEFORE the observation date
    # (if paid before, nothing to predict)
    temp = temp[
        (temp['full_pay_date'].isna()) |  # not paid at all
        (temp['full_pay_date'] >= temp['observation_date'])  # paid on or after observation
    ]

    temp['obs_offset_days'] = offset
    observations.append(temp)

obs_df = pd.concat(observations, ignore_index=True)
print(f"Total observation rows: {len(obs_df):,}")
print(f"Observation offset distribution:")
print(obs_df['obs_offset_days'].value_counts().sort_index())

Total observation rows: 410,854
Observation offset distribution:
obs_offset_days
0     179062
15    124126
30     71569
60     36097
Name: count, dtype: int64


# Define the Target Variable

In [6]:
"""
STEP 3: TARGET VARIABLE

Target: paid_within_90_days
  1 = invoice was fully paid within 90 days of DUE DATE
  0 = not fully paid within 90 days

This is computed from ground truth (actual payment dates),
but during feature engineering, features will ONLY use data
available on or before the observation_date.
"""

# paid_within_90_days: was it fully paid within 90 days of due date?
obs_df['target_deadline'] = obs_df['due_date'] + timedelta(days=90)
obs_df['paid_within_90d'] = (
    obs_df['is_fully_paid'] &
    (obs_df['full_pay_date'] <= obs_df['target_deadline'])
).astype(int)

print(f"Target distribution:")
print(obs_df['paid_within_90d'].value_counts())
print(f"\nTarget balance: {obs_df['paid_within_90d'].mean():.1%} positive (paid)")

Target distribution:
paid_within_90d
1    258849
0    152005
Name: count, dtype: int64

Target balance: 63.0% positive (paid)


# Invoice-Level Features (Static)

In [7]:
"""
STEP 4: INVOICE-LEVEL FEATURES
These come directly from the invoice itself. No leakage risk.
"""

# Merge customer info
obs_df = obs_df.merge(
    customers[['customer_id', 'customer_segment', 'industry', 'region',
               'payment_terms_days', 'credit_limit', 'customer_since',
               'customer_risk_tier']],
    on='customer_id', how='left', suffixes=('', '_cust')
)

# ── Invoice features ──
obs_df['log_invoice_amount'] = np.log1p(obs_df['invoice_amount'])
obs_df['days_past_due'] = (obs_df['observation_date'] - obs_df['due_date']).dt.days
obs_df['days_since_invoice'] = (obs_df['observation_date'] - obs_df['invoice_date']).dt.days

# Aging bucket
bins = [-999, 0, 30, 60, 90, 120, 999]
labels = [0, 1, 2, 3, 4, 5]  # numerical for model
obs_df['aging_bucket'] = pd.cut(obs_df['days_past_due'], bins=bins, labels=labels).astype(float)

# Percent of payment term elapsed
obs_df['pct_term_elapsed'] = obs_df['days_since_invoice'] / obs_df['payment_terms_days'].clip(lower=1)

# Has PO number (customers without PO = higher risk in practice)
obs_df['has_po_number'] = obs_df['po_number'].notna().astype(int)

# Is recurring invoice
obs_df['is_recurring'] = obs_df['is_recurring'].astype(int)

# Customer tenure at observation date (months)
obs_df['customer_tenure_months'] = (
    (obs_df['observation_date'] - obs_df['customer_since']).dt.days / 30.44
).clip(lower=0)

# Amount relative to credit limit
obs_df['amount_vs_credit_limit'] = obs_df['invoice_amount'] / obs_df['credit_limit'].clip(lower=1)

print("Invoice-level features created:")
print("  log_invoice_amount, days_past_due, days_since_invoice,")
print("  aging_bucket, pct_term_elapsed, has_po_number,")
print("  is_recurring, customer_tenure_months, amount_vs_credit_limit")

Invoice-level features created:
  log_invoice_amount, days_past_due, days_since_invoice,
  aging_bucket, pct_term_elapsed, has_po_number,
  is_recurring, customer_tenure_months, amount_vs_credit_limit


# Customer Historical Payment Features (Rolling)

In [8]:
"""
STEP 5: CUSTOMER HISTORICAL PAYMENT FEATURES

These are the most important features — and the most dangerous for leakage.

RULE: For each observation row, we ONLY use payments that happened
BEFORE the observation_date. This is non-negotiable.

We compute rolling features over 6-month and 12-month lookback windows.
"""

# Pre-build a payment history lookup: each payment + its invoice due_date
pay_hist = payments.merge(
    invoices[['invoice_id', 'due_date', 'invoice_amount']],
    on='invoice_id', how='left'
)
pay_hist['days_to_pay'] = (pay_hist['payment_date'] - pay_hist['due_date']).dt.days
pay_hist['paid_on_time'] = (pay_hist['days_to_pay'] <= 0).astype(int)

def compute_customer_payment_features(obs_row, pay_hist, lookback_months=6):
    """
    Compute rolling payment features for one customer as of one observation date.
    Only uses payments made BEFORE the observation date.
    """
    obs_date = obs_row['observation_date']
    cust_id = obs_row['customer_id']
    cutoff = obs_date - pd.DateOffset(months=lookback_months)

    # Filter: this customer's payments, before observation date, within lookback
    mask = (
        (pay_hist['customer_id'] == cust_id) &
        (pay_hist['payment_date'] < obs_date) &
        (pay_hist['payment_date'] >= cutoff)
    )
    hist = pay_hist.loc[mask]

    if len(hist) == 0:
        return pd.Series({
            f'avg_days_to_pay_{lookback_months}m': np.nan,
            f'std_days_to_pay_{lookback_months}m': np.nan,
            f'median_days_to_pay_{lookback_months}m': np.nan,
            f'on_time_rate_{lookback_months}m': np.nan,
            f'payment_count_{lookback_months}m': 0,
            f'worst_delay_{lookback_months}m': np.nan,
        })

    return pd.Series({
        f'avg_days_to_pay_{lookback_months}m': hist['days_to_pay'].mean(),
        f'std_days_to_pay_{lookback_months}m': hist['days_to_pay'].std(),
        f'median_days_to_pay_{lookback_months}m': hist['days_to_pay'].median(),
        f'on_time_rate_{lookback_months}m': hist['paid_on_time'].mean(),
        f'payment_count_{lookback_months}m': len(hist),
        f'worst_delay_{lookback_months}m': hist['days_to_pay'].max(),
    })

# ──────────────────────────────────────────
# OPTIMIZED: Pre-compute per (customer, month) to avoid row-by-row loop
# ──────────────────────────────────────────

print("Computing customer payment history features...")
print("(This may take 2-3 minutes...)\n")

# Strategy: for each unique (customer_id, observation_month), compute once
obs_df['obs_month'] = obs_df['observation_date'].dt.to_period('M')
unique_obs = obs_df[['customer_id', 'observation_date', 'obs_month']].drop_duplicates(
    subset=['customer_id', 'obs_month']
).reset_index(drop=True)

print(f"Unique (customer, month) combinations: {len(unique_obs):,}")

# Compute 6-month rolling features
results_6m = []
for i, row in unique_obs.iterrows():
    result = compute_customer_payment_features(row, pay_hist, lookback_months=6)
    results_6m.append(result)
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i+1:,}/{len(unique_obs):,}...")

features_6m = pd.DataFrame(results_6m)
features_6m['customer_id'] = unique_obs['customer_id'].values
features_6m['obs_month'] = unique_obs['obs_month'].values

# Merge back to observation dataframe
obs_df = obs_df.merge(features_6m, on=['customer_id', 'obs_month'], how='left')

# ── Payment Consistency Index ──
# Low = consistent (predictable), High = erratic
obs_df['payment_consistency_idx'] = (
    obs_df['std_days_to_pay_6m'] / (obs_df['avg_days_to_pay_6m'].abs() + 1)
)

print("\nCustomer payment history features created:")
print("  avg_days_to_pay_6m, std_days_to_pay_6m, median_days_to_pay_6m,")
print("  on_time_rate_6m, payment_count_6m, worst_delay_6m,")
print("  payment_consistency_idx")
print(f"\nNull rates (expected for new customers with no history):")
print(obs_df[['avg_days_to_pay_6m', 'on_time_rate_6m', 'payment_count_6m']].isnull().mean().round(3))

Computing customer payment history features...
(This may take 2-3 minutes...)

Unique (customer, month) combinations: 88,291
  Processed 10,000/88,291...
  Processed 20,000/88,291...
  Processed 30,000/88,291...
  Processed 40,000/88,291...
  Processed 50,000/88,291...
  Processed 60,000/88,291...
  Processed 70,000/88,291...
  Processed 80,000/88,291...

Customer payment history features created:
  avg_days_to_pay_6m, std_days_to_pay_6m, median_days_to_pay_6m,
  on_time_rate_6m, payment_count_6m, worst_delay_6m,
  payment_consistency_idx

Null rates (expected for new customers with no history):
avg_days_to_pay_6m    0.059
on_time_rate_6m       0.059
payment_count_6m      0.000
dtype: float64


# Dispute Features

In [9]:
"""
STEP 6: DISPUTE FEATURES
Only use disputes raised BEFORE the observation date.
"""

def compute_dispute_features(obs_df, disputes, invoices):
    """Vectorized dispute feature computation."""

    # ── Feature 1: Does THIS specific invoice have an active dispute? ──
    # Active = dispute raised before obs_date AND (not resolved OR resolved after obs_date)
    disp_inv = disputes[['invoice_id', 'dispute_date', 'resolution_date', 'dispute_amount']].copy()

    obs_with_disp = obs_df[['invoice_id', 'observation_date']].merge(
        disp_inv, on='invoice_id', how='left'
    )
    obs_with_disp['has_active_dispute'] = (
        obs_with_disp['dispute_date'].notna() &
        (obs_with_disp['dispute_date'] <= obs_with_disp['observation_date']) &
        (
            obs_with_disp['resolution_date'].isna() |
            (obs_with_disp['resolution_date'] > obs_with_disp['observation_date'])
        )
    ).astype(int)

    # Take max per invoice-observation (in case multiple disputes)
    has_dispute = obs_with_disp.groupby(['invoice_id', 'observation_date'])[
        'has_active_dispute'
    ].max().reset_index()

    obs_df = obs_df.merge(has_dispute, on=['invoice_id', 'observation_date'], how='left')
    obs_df['has_active_dispute'] = obs_df['has_active_dispute'].fillna(0).astype(int)

    # ── Feature 2: Customer dispute rate in last 12 months ──
    disp_cust = disputes.merge(
        obs_df[['customer_id', 'observation_date']].drop_duplicates(),
        on='customer_id', how='inner'
    )
    disp_cust['in_window'] = (
        (disp_cust['dispute_date'] < disp_cust['observation_date']) &
        (disp_cust['dispute_date'] >= disp_cust['observation_date'] - pd.DateOffset(months=12))
    )
    cust_disp_count = disp_cust[disp_cust['in_window']].groupby(
        ['customer_id', 'observation_date']
    ).size().reset_index(name='dispute_count_12m')

    # Total invoices per customer in same window
    inv_cust = invoices.merge(
        obs_df[['customer_id', 'observation_date']].drop_duplicates(),
        on='customer_id', how='inner'
    )
    inv_cust['in_window'] = (
        (inv_cust['invoice_date'] < inv_cust['observation_date']) &
        (inv_cust['invoice_date'] >= inv_cust['observation_date'] - pd.DateOffset(months=12))
    )
    cust_inv_count = inv_cust[inv_cust['in_window']].groupby(
        ['customer_id', 'observation_date']
    ).size().reset_index(name='inv_count_12m')

    cust_disp_rate = cust_disp_count.merge(cust_inv_count, on=['customer_id', 'observation_date'], how='outer')
    cust_disp_rate['dispute_count_12m'] = cust_disp_rate['dispute_count_12m'].fillna(0)
    cust_disp_rate['inv_count_12m'] = cust_disp_rate['inv_count_12m'].fillna(1)
    cust_disp_rate['customer_dispute_rate_12m'] = (
        cust_disp_rate['dispute_count_12m'] / cust_disp_rate['inv_count_12m']
    )

    obs_df = obs_df.merge(
        cust_disp_rate[['customer_id', 'observation_date', 'customer_dispute_rate_12m']],
        on=['customer_id', 'observation_date'], how='left'
    )
    obs_df['customer_dispute_rate_12m'] = obs_df['customer_dispute_rate_12m'].fillna(0)

    return obs_df

print("Computing dispute features...")
obs_df = compute_dispute_features(obs_df, disputes, invoices)
print("  has_active_dispute, customer_dispute_rate_12m")
print(f"  Active dispute rate: {obs_df['has_active_dispute'].mean():.2%}")
print(f"  Avg customer dispute rate: {obs_df['customer_dispute_rate_12m'].mean():.2%}")

Computing dispute features...
  has_active_dispute, customer_dispute_rate_12m
  Active dispute rate: 3.36%
  Avg customer dispute rate: 8.10%


# Dunning / Contact Features

In [10]:
"""
STEP 7: DUNNING / CONTACT FEATURES
Only use contacts made BEFORE the observation date.
"""

print("Computing dunning features...")

# Filter contacts to before observation date per invoice
# Pre-build a lookup
dunning_sorted = dunning.sort_values('contact_date')

# For efficiency: aggregate per invoice first, then filter by date
def compute_dunning_features_batch(obs_df, dunning):
    """Compute dunning features for all observation rows."""

    # Merge observations with their invoice's contacts
    obs_contacts = obs_df[['invoice_id', 'observation_date']].merge(
        dunning[['invoice_id', 'contact_date', 'contact_type', 'contact_outcome',
                 'promised_pay_date', 'contact_sequence']],
        on='invoice_id', how='left'
    )

    # Filter: only contacts BEFORE observation date
    obs_contacts = obs_contacts[
        obs_contacts['contact_date'] < obs_contacts['observation_date']
    ]

    # Aggregate per (invoice_id, observation_date)
    if len(obs_contacts) == 0:
        obs_df['contacts_to_date'] = 0
        obs_df['days_since_last_contact'] = np.nan
        obs_df['has_promise_to_pay'] = 0
        obs_df['phone_contact_count'] = 0
        obs_df['max_contact_sequence'] = 0
        return obs_df

    grouped = obs_contacts.groupby(['invoice_id', 'observation_date']).agg(
        contacts_to_date=('contact_date', 'count'),
        last_contact_date=('contact_date', 'max'),
        has_promise_to_pay=('contact_outcome', lambda x: int(('Promise to Pay' in x.values))),
        phone_contact_count=('contact_type', lambda x: (x == 'Phone').sum()),
        max_contact_sequence=('contact_sequence', 'max'),
    ).reset_index()

    obs_df = obs_df.merge(grouped, on=['invoice_id', 'observation_date'], how='left')

    # Fill nulls (invoices with no contacts yet)
    obs_df['contacts_to_date'] = obs_df['contacts_to_date'].fillna(0).astype(int)
    obs_df['has_promise_to_pay'] = obs_df['has_promise_to_pay'].fillna(0).astype(int)
    obs_df['phone_contact_count'] = obs_df['phone_contact_count'].fillna(0).astype(int)
    obs_df['max_contact_sequence'] = obs_df['max_contact_sequence'].fillna(0).astype(int)

    # Days since last contact
    obs_df['days_since_last_contact'] = (
        obs_df['observation_date'] - obs_df['last_contact_date']
    ).dt.days
    obs_df.drop(columns=['last_contact_date'], inplace=True, errors='ignore')

    return obs_df

obs_df = compute_dunning_features_batch(obs_df, dunning)

print("  contacts_to_date, days_since_last_contact, has_promise_to_pay,")
print("  phone_contact_count, max_contact_sequence")
print(f"  Avg contacts per observation: {obs_df['contacts_to_date'].mean():.1f}")
print(f"  PTP rate: {obs_df['has_promise_to_pay'].mean():.2%}")

Computing dunning features...
  contacts_to_date, days_since_last_contact, has_promise_to_pay,
  phone_contact_count, max_contact_sequence
  Avg contacts per observation: 1.7
  PTP rate: 11.55%


# Portfolio Pressure / Credit Stress Features

In [12]:
"""
STEP 8: PORTFOLIO PRESSURE & CREDIT STRESS FEATURES (FAST VERSION)
"""

print("Computing portfolio pressure features (optimized)...")

# Total paid per invoice
paid_per_inv = payments.groupby('invoice_id')['payment_amount'].sum().reset_index(name='total_paid_amt')
inv_balances = invoices[['invoice_id', 'customer_id', 'invoice_date', 'invoice_amount']].merge(
    paid_per_inv, on='invoice_id', how='left'
)
inv_balances['total_paid_amt'] = inv_balances['total_paid_amt'].fillna(0)
inv_balances['balance'] = inv_balances['invoice_amount'] - inv_balances['total_paid_amt']

# Keep only invoices with remaining balance
inv_open = inv_balances[inv_balances['balance'] > inv_balances['invoice_amount'] * 0.01].copy()

# Get unique (customer, obs_month) keys
unique_keys = obs_df[['customer_id', 'obs_month', 'observation_date']].drop_duplicates(
    subset=['customer_id', 'obs_month']
).reset_index(drop=True)

# Merge: for each (customer, obs_month), get all their open invoices
merged = unique_keys.merge(inv_open, on='customer_id', how='inner')

# Filter: invoice must be issued BEFORE observation date
merged = merged[merged['invoice_date'] <= merged['observation_date']]

# Aggregate per (customer, obs_month)
portfolio = merged.groupby(['customer_id', 'obs_month']).agg(
    open_invoice_count=('invoice_id', 'count'),
    total_open_ar=('balance', 'sum'),
).reset_index()

# Merge back
obs_df = obs_df.merge(portfolio, on=['customer_id', 'obs_month'], how='left')
obs_df['open_invoice_count'] = obs_df['open_invoice_count'].fillna(0).astype(int)
obs_df['total_open_ar'] = obs_df['total_open_ar'].fillna(0)

# Credit utilization
obs_df['credit_utilization'] = (
    obs_df['total_open_ar'] / obs_df['credit_limit'].clip(lower=1)
).clip(upper=5.0)

print("Done!")
print(f"  Avg open invoices per customer: {obs_df['open_invoice_count'].mean():.1f}")
print(f"  Avg credit utilization: {obs_df['credit_utilization'].mean():.2f}")

Computing portfolio pressure features (optimized)...
Done!
  Avg open invoices per customer: 5.4
  Avg credit utilization: 0.36


# Seasonal / Calendar Features

In [13]:
"""
STEP 9: SEASONAL AND CALENDAR FEATURES
Capture Q1 effects, quarter-end patterns, etc.
"""

obs_df['due_month'] = obs_df['due_date'].dt.month
obs_df['due_quarter'] = obs_df['due_date'].dt.quarter
obs_df['is_q1'] = (obs_df['due_quarter'] == 1).astype(int)
obs_df['is_quarter_end_month'] = obs_df['due_month'].isin([3, 6, 9, 12]).astype(int)
obs_df['is_year_end'] = (obs_df['due_month'] == 12).astype(int)

print("Seasonal features created:")
print("  due_month, due_quarter, is_q1, is_quarter_end_month, is_year_end")
print(f"  Q1 invoices: {obs_df['is_q1'].mean():.1%}")

Seasonal features created:
  due_month, due_quarter, is_q1, is_quarter_end_month, is_year_end
  Q1 invoices: 20.2%


# Customer Health Score (Composite)

In [14]:
"""
STEP 10: CUSTOMER HEALTH SCORE

A composite feature that rolls up multiple signals into one 0-1 score.
This is a standout feature for interviews — shows you can synthesize
multiple risk signals into a single actionable metric.
"""

def normalize_series(s, higher_is_better=True):
    """Normalize to 0-1 range. Handle NaN."""
    s_clean = s.fillna(s.median())
    s_min, s_max = s_clean.min(), s_clean.max()
    if s_max == s_min:
        return pd.Series(0.5, index=s.index)
    normed = (s_clean - s_min) / (s_max - s_min)
    if not higher_is_better:
        normed = 1 - normed
    return normed

obs_df['health_score'] = (
    0.25 * normalize_series(obs_df['on_time_rate_6m'], higher_is_better=True) +
    0.20 * normalize_series(obs_df['credit_utilization'], higher_is_better=False) +
    0.15 * normalize_series(obs_df['payment_consistency_idx'], higher_is_better=False) +
    0.15 * normalize_series(obs_df['customer_dispute_rate_12m'], higher_is_better=False) +
    0.15 * normalize_series(obs_df['payment_count_6m'], higher_is_better=True) +
    0.10 * normalize_series(obs_df['customer_tenure_months'], higher_is_better=True)
).clip(0, 1)

print(f"Customer Health Score created (0=worst, 1=best):")
print(f"  Mean: {obs_df['health_score'].mean():.3f}")
print(f"  Median: {obs_df['health_score'].median():.3f}")
print(f"  Std: {obs_df['health_score'].std():.3f}")
print(obs_df['health_score'].describe())

Customer Health Score created (0=worst, 1=best):
  Mean: 0.559
  Median: 0.556
  Std: 0.058
count    410854.000000
mean          0.559343
std           0.057501
min           0.288235
25%           0.521742
50%           0.556188
75%           0.590848
max           0.807074
Name: health_score, dtype: float64


# Encode Categorical Variables

In [15]:
"""
STEP 11: ENCODE CATEGORICAL VARIABLES
Convert segment, industry, region to numerical for the model.
"""

# One-hot encode customer segment (3 categories → 2 dummies)
segment_dummies = pd.get_dummies(obs_df['customer_segment'], prefix='seg', drop_first=True)
obs_df = pd.concat([obs_df, segment_dummies], axis=1)

# One-hot encode region (4 categories → 3 dummies)
region_dummies = pd.get_dummies(obs_df['region'], prefix='region', drop_first=True)
obs_df = pd.concat([obs_df, region_dummies], axis=1)

# Risk tier → ordinal
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
obs_df['risk_tier_encoded'] = obs_df['customer_risk_tier'].map(risk_map)

# Industry → target encoding would be ideal, but for V1 just use a few binary flags
obs_df['is_manufacturing'] = (obs_df['industry'] == 'Manufacturing').astype(int)
obs_df['is_it_services'] = (obs_df['industry'] == 'IT Services').astype(int)
obs_df['is_retail'] = (obs_df['industry'] == 'Retail').astype(int)

print("Categorical encoding complete.")
print(f"  Segment dummies: {list(segment_dummies.columns)}")
print(f"  Region dummies: {list(region_dummies.columns)}")
print(f"  Risk tier: ordinal (0=Low, 1=Medium, 2=High)")

Categorical encoding complete.
  Segment dummies: ['seg_Mid-Market', 'seg_SMB']
  Region dummies: ['region_North', 'region_South', 'region_West']
  Risk tier: ordinal (0=Low, 1=Medium, 2=High)


# Select Final Features and Save

In [16]:
"""
STEP 12: SELECT FINAL FEATURES AND SAVE MODELING DATASET

This cell defines the exact feature list for the model.
Anything NOT in this list stays out of the model.
"""

# ── Define feature columns ──
FEATURE_COLUMNS = [
    # Invoice-level
    'invoice_amount', 'log_invoice_amount', 'payment_terms_days',
    'days_past_due', 'days_since_invoice', 'aging_bucket', 'pct_term_elapsed',
    'has_po_number', 'is_recurring', 'line_item_count',

    # Customer historical payment (rolling 6m)
    'avg_days_to_pay_6m', 'std_days_to_pay_6m', 'median_days_to_pay_6m',
    'on_time_rate_6m', 'payment_count_6m', 'worst_delay_6m',
    'payment_consistency_idx',

    # Dispute
    'has_active_dispute', 'customer_dispute_rate_12m',

    # Dunning / contacts
    'contacts_to_date', 'days_since_last_contact', 'has_promise_to_pay',
    'phone_contact_count', 'max_contact_sequence',

    # Portfolio pressure / credit stress
    'open_invoice_count', 'total_open_ar', 'credit_utilization',
    'amount_vs_credit_limit',

    # Relationship
    'customer_tenure_months',

    # Seasonal
    'is_q1', 'is_quarter_end_month', 'is_year_end',

    # Composite
    'health_score',

    # Encoded categoricals
    'risk_tier_encoded',
    'seg_Mid-Market', 'seg_SMB',
    'region_North', 'region_South', 'region_West',
    'is_manufacturing', 'is_it_services', 'is_retail',
]

# Columns to keep for reference (not features)
ID_COLUMNS = ['invoice_id', 'customer_id', 'observation_date', 'obs_offset_days',
              'invoice_date', 'due_date']
TARGET_COLUMN = 'paid_within_90d'

# ── Build final dataset ──
# Check which features actually exist (in case some encoding names differ)
available_features = [c for c in FEATURE_COLUMNS if c in obs_df.columns]
missing_features = [c for c in FEATURE_COLUMNS if c not in obs_df.columns]
if missing_features:
    print(f"⚠ Missing features (will be skipped): {missing_features}")

modeling_df = obs_df[ID_COLUMNS + available_features + [TARGET_COLUMN]].copy()

# ── Handle remaining NaN values ──
# For rolling features: NaN means customer had no history → fill with -1 (flag value)
# For days_since_last_contact: NaN means no contact made → fill with 999
nan_fills = {
    'avg_days_to_pay_6m': -1,
    'std_days_to_pay_6m': -1,
    'median_days_to_pay_6m': -1,
    'on_time_rate_6m': -1,
    'worst_delay_6m': -1,
    'payment_consistency_idx': -1,
    'days_since_last_contact': 999,
    'aging_bucket': 0,
}
for col, fill_val in nan_fills.items():
    if col in modeling_df.columns:
        before_nulls = modeling_df[col].isnull().sum()
        modeling_df[col] = modeling_df[col].fillna(fill_val)
        if before_nulls > 0:
            print(f"  Filled {before_nulls:,} NaN in {col} with {fill_val}")

# Any remaining NaN → fill with 0
remaining_nulls = modeling_df[available_features].isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if len(remaining_nulls) > 0:
    print(f"\nRemaining nulls filled with 0:")
    print(remaining_nulls)
    modeling_df[available_features] = modeling_df[available_features].fillna(0)

print(f"\n{'='*60}")
print(f"MODELING DATASET READY")
print(f"{'='*60}")
print(f"  Shape: {modeling_df.shape}")
print(f"  Features: {len(available_features)}")
print(f"  Target: {TARGET_COLUMN}")
print(f"  Target balance: {modeling_df[TARGET_COLUMN].mean():.1%} positive")
print(f"  Null values remaining: {modeling_df[available_features].isnull().sum().sum()}")

  Filled 24,096 NaN in avg_days_to_pay_6m with -1
  Filled 38,085 NaN in std_days_to_pay_6m with -1
  Filled 24,096 NaN in median_days_to_pay_6m with -1
  Filled 24,096 NaN in on_time_rate_6m with -1
  Filled 24,096 NaN in worst_delay_6m with -1
  Filled 38,085 NaN in payment_consistency_idx with -1
  Filled 179,916 NaN in days_since_last_contact with 999

MODELING DATASET READY
  Shape: (410854, 49)
  Features: 42
  Target: paid_within_90d
  Target balance: 63.0% positive
  Null values remaining: 0


# Save and Validate

In [20]:
"""
STEP 13: SAVE AND VALIDATE
"""

OUTPUT_PATH = 'C:/Users/Manu/Desktop/Project 2 AR Collections Intelligence Engine/Raw Data/'
import os
os.makedirs(OUTPUT_PATH, exist_ok=True)

modeling_df.to_csv(f'{OUTPUT_PATH}modeling_dataset.csv', index=False)
print(f"Saved to: {OUTPUT_PATH}modeling_dataset.csv")
print(f"File size: {os.path.getsize(f'{OUTPUT_PATH}modeling_dataset.csv') / (1024*1024):.1f} MB")

# ── Final validation ──
print(f"\n{'='*60}")
print("FINAL VALIDATION")
print(f"{'='*60}")

# 1. No leakage check: target should NOT be perfectly predicted by any single feature
print("\n1. Feature-Target Correlations (check for leakage):")
corrs = modeling_df[available_features + [TARGET_COLUMN]].corr()[TARGET_COLUMN].drop(TARGET_COLUMN)
corrs_sorted = corrs.abs().sort_values(ascending=False)
for feat, corr in corrs_sorted.head(15).items():
    flag = " ⚠ INVESTIGATE" if corr > 0.5 else ""
    print(f"   {feat:35s}: {corr:.4f}{flag}")

# 2. No features > 0.5 correlation (would suggest leakage)
max_corr = corrs_sorted.iloc[0]
if max_corr > 0.5:
    print(f"\n⚠ WARNING: Feature '{corrs_sorted.index[0]}' has {max_corr:.3f} correlation with target!")
    print("  This MIGHT indicate leakage. Investigate before modeling.")
else:
    print(f"\n✅ No single feature has >0.5 correlation with target. No leakage detected.")

# 3. Feature summary stats
print(f"\n2. Feature Summary Statistics:")
print(modeling_df[available_features].describe().round(2).to_string())

# 4. Observation date range for train/val/test planning
print(f"\n3. Observation Date Range:")
print(f"   Min: {modeling_df['observation_date'].min()}")
print(f"   Max: {modeling_df['observation_date'].max()}")
print(f"   Unique dates: {modeling_df['observation_date'].nunique():,}")

print(f"\n{'='*60}")
print("READY FOR NOTEBOOK 04: CLASSIFICATION MODEL")
print(f"{'='*60}")
print(f"""
In Notebook 04, load this dataset with:
  modeling_df = pd.read_csv('{OUTPUT_PATH}modeling_dataset.csv',
                            parse_dates=['observation_date', 'invoice_date', 'due_date'])

Then split by time:
  Train: observation_date <= '2023-06-30'
  Val:   observation_date in Jul-Sep 2023
  Test:  observation_date in Oct-Dec 2023

Feature columns: {len(available_features)} features
Target column:   {TARGET_COLUMN}
""")

Saved to: C:/Users/Manu/Desktop/Project 2 AR Collections Intelligence Engine/Raw Data/modeling_dataset.csv
File size: 132.8 MB

FINAL VALIDATION

1. Feature-Target Correlations (check for leakage):
   days_past_due                      : 0.3980
   pct_term_elapsed                   : 0.3782
   max_contact_sequence               : 0.3608
   contacts_to_date                   : 0.3497
   aging_bucket                       : 0.3439
   phone_contact_count                : 0.3030
   days_since_invoice                 : 0.2999
   open_invoice_count                 : 0.2743
   days_since_last_contact            : 0.2327
   credit_utilization                 : 0.1621
   risk_tier_encoded                  : 0.1591
   median_days_to_pay_6m              : 0.1517
   avg_days_to_pay_6m                 : 0.1495
   has_promise_to_pay                 : 0.1429
   total_open_ar                      : 0.1427

✅ No single feature has >0.5 correlation with target. No leakage detected.

2. Feature Summary S